# SIMD 抽象硬件架构简介

## 学习目标

Ascend C 面向 AI Core 编程，但开发者不需要一开始就记住所有硬件实现细节。抽象硬件架构把不同产品和架构版本中的共性提炼出来，用统一视角说明计算单元、存储单元、搬运单元和同步机制如何协同工作。本节的重点不是穷尽 AI Core 的所有硬件规格，而是为理解 SIMD 编程模型、分析数据流和后续编写算子建立必要的硬件基础。

完成本节学习后，你应该能够：

- 识别抽象硬件架构图中的核心组件，包括计算单元、存储单元和搬运单元，并说明它们在算子执行中的基本职责；
- 结合数据搬运、片上计算和同步依赖，解读 Scalar、Vector、Cube、Global Memory、Local Memory、DMA 等核心组件在算子执行中的作用；
- 了解 NPU 架构版本 2201 与 3510 在抽象硬件架构视角下的主要差异，并知道具体规格需要查阅对应官方文档。


## 1. 抽象架构图中的核心组件

下图为通用 NPU 架构图，展示 AI Core 抽象硬件架构中主要组件的位置关系和协同方式。

<img src="./images/abstract-hardware-arch.png" alt="NPU 架构版本 2201 的 AI Core 抽象硬件架构" width="720px">

阅读这张图时，可以先按职责把核心组件分成三类：计算单元、存储单元和搬运单元。

| 类别 | 典型组件 | 基本职责 |
| --- | --- | --- |
| 计算单元 | Scalar、Vector、Cube | 负责控制逻辑、矢量计算和矩阵计算。 |
| 存储单元 | Global Memory、Local Memory | 保存输入、输出、片上分片数据和中间结果。 |
| 搬运单元 | DMA（MTE1、MTE2、MTE3 等） | 在不同存储层级之间转移数据，让数据靠近计算单元。 |

### 1.1 计算单元

计算单元决定“谁来组织任务”和“谁来执行计算”。SIMD 入门阶段最常接触的是 Scalar 与 Vector，矩阵类算子会进一步涉及 Cube。

| 组件 | 主要职责 | 在算子执行中的作用 |
| --- | --- | --- |
| Scalar | 执行地址计算、循环控制、分支控制等标量工作，并向 Vector、Cube、DMA 等单元发射任务。 | 负责组织当前 AI Core 上的控制流程，把计算、搬运和同步任务串联起来。 |
| Vector | 执行矢量运算，适合对一批元素做相同或相近的逐元素计算。 | 负责处理 SIMD 场景中的批量逐元素计算，是矢量类算子的主要计算单元。 |
| Cube | 执行矩阵运算，适合矩阵乘、卷积等高密度矩阵类计算。 | 负责处理矩阵类高密度计算，通常与 L1、L0 等片上存储层级配合使用。 |

Scalar 可以理解为片上的控制者，但不要把它理解成负责所有大批量计算的 CPU。大量逐元素计算通常交给 Vector，矩阵类计算通常交给 Cube，Scalar 主要负责把这些任务组织起来。

### 1.2 存储单元

存储单元需要重点区分 Global Memory（GM）和 Local Memory（LM）。其中 GM 用于保存 Device 侧全局可见的数据，而 LM 是 AI Core 片上的本地存储，用于承接当前计算任务需要访问的数据分片和中间结果。

为什么要区分 GM 和 LM？可以先从一个通用计算机常识理解：存储通常不是只有一种。离计算单元越近的存储，访问通常越快，但容量更小；离计算单元越远的存储，容量通常更大，但访问开销更高。因此，很多计算系统都会把“长期保存完整数据的位置”和“临时参与当前计算的位置”区分开。

在 AI Core 中，GM 和 LM 就承担了这样的分工。GM 更适合保存完整输入、完整输出，以及多个 AI Core 或不同任务都可能需要访问的数据；LM 位于 AI Core 片上，更适合保存当前 AI Core 正在处理的一小段数据和中间结果。换句话说，GM 关注“完整数据放在哪里”，LM 关注“当前这一小段数据在哪里被高效计算”。

因此，一个 SIMD 算子通常不会直接围绕 GM 中的完整 Tensor 完成所有计算，而是按分片组织执行：完整输入输出放在 GM 中，当前 AI Core 要处理的数据通过 DMA 搬入 LM，Vector 或 Cube 基于 LM 中的数据执行计算，结果再写回 GM。区分 GM 和 LM，可以帮助开发者先看清数据流：数据从哪里来、搬到哪里算、算完以后写回哪里。

可以先按下面的关系理解 GM 和 LM。

| 存储类别 | 位置和可见范围 | 保存的数据 | 在执行中的作用 | 入门理解 |
| --- | --- | --- | --- | --- |
| Global Memory（GM） | Device 侧全局存储，容量更大，可作为多个 AI Core 或不同任务之间的数据汇合位置。 | 算子的完整输入、完整输出，以及需要在片上计算之外长期保留的数据。 | 作为数据搬入 LM 的来源，也是计算结果写回后的最终保存位置。 | GM 更像完整 Tensor 的全局存放区。 |
| Local Memory（LM） | AI Core 片上本地存储，离计算单元更近，容量通常小于 GM。 | 当前 AI Core、当前分片要处理的数据，以及片上计算过程中产生的中间结果。 | 为 Vector、Cube 等计算单元提供更近的数据访问位置，承接“搬入、计算、搬出”中的片上阶段。 | LM 更像当前计算分片的片上工作区。 |

Local Memory 不是单一硬件资源，而是对 AI Core 片上本地存储的抽象称呼。不同架构版本和不同计算方式会对应不同的片上存储层级。本节先把它作为“片上工作区”理解，后续章节再结合具体编程方式展开对应的存储资源和接口。

### 1.3 搬运单元

DMA（Direct Memory Access）搬运单元负责数据搬运，包括 GM 与 LM 之间的数据搬入、搬出，以及不同 Local Memory 层级之间的数据流转。常见搬运单元包括 MTE1、MTE2、MTE3 和 FixPipe。

从算子执行角度看，搬运单元的作用是把数据移动到合适的位置：输入数据需要从 GM 进入片上存储，计算结果需要从片上存储写回 GM。搬运和计算之间存在依赖时，需要同步信号保证先后关系正确。


## 2. 从计算数据流理解架构图

理解了抽象架构中的核心组件之后，可以先从计算数据流这一条主线阅读架构图：数据从 GM 进入片上存储，在计算单元中完成计算，再将结果写回 GM。

<img src="./images/simd-arch-data-flow.png" alt="AI Core 抽象硬件架构中的计算数据流" width="820px">

计算数据流关注的是“数据从哪里来、在哪里算、结果写回哪里”。完整输入输出通常位于 GM，当前分片通过 DMA 搬入 LM，Vector 或 Cube 基于片上数据执行计算，计算结果再经过 DMA 写回 GM。

可以把这条主线抽象成四个阶段：

| 计算数据流阶段 | 关联架构组件 | 作用 |
| --- | --- | --- |
| GM 中保存完整输入 | Global Memory | 保存算子的完整输入数据。 |
| 当前分片搬入片上存储 | DMA、Local Memory | DMA 将当前 AI Core 负责的数据搬入更靠近计算单元的片上存储。 |
| 片上完成计算 | Vector / Cube | 计算单元基于片上数据执行矢量或矩阵计算。 |
| 结果写回 GM | DMA、Global Memory | DMA 将片上计算结果写回全局输出位置。 |

阅读架构图时，先抓住“GM -> LM -> Vector/Cube -> GM”这条路径，就能把存储位置、搬运单元和计算单元串起来。后续学习更复杂的算子时，也可以先判断数据当前位于 GM 还是片上存储、由哪个计算单元处理、最后写回到哪里。


## 3. 架构版本差异：2201 与 3510

不同 NPU 架构版本会在计算单元、存储层级、数据搬运路径和可用编程方式上有差异。对入门学习来说，本节不展开硬件规格细节，只需要知道：前面建立的“计算、存储、搬运、同步”这套抽象视角仍然适用，但具体能力和最佳写法要结合目标架构版本理解。

下图展示 NPU 架构版本 3510 的抽象硬件架构，可以把它和前面的 2201 架构图对照阅读。

<img src="./images/abstract-hardware-arch-3510.png" alt="NPU 架构版本 3510 的 AI Core 抽象硬件架构" width="760px">

简单理解 2201 与 3510 的差异时，可以先抓住三点：

- **计算方式更丰富**：3510 在 SIMD Vector 编程视角下引入 Regbase 等能力，可利用寄存器承载部分中间结果；
- **存储和搬运细节会变化**：不同架构版本的片上存储层级、搬运单元能力和数据流转路径可能不同；
- **编程模型会扩展**：3510 除 SIMD 编程外，还会涉及 SIMT 以及 SIMD/SIMT 混合编程等更丰富的方式。

因此，阅读不同架构版本的资料或示意图时，不需要一开始记住所有硬件差异。更实用的做法是先用统一抽象定位问题：数据在 GM 还是片上存储？由哪个计算单元处理？是否需要 DMA 搬运？任务之间有什么同步依赖？具体到某个产品型号和架构版本时，再查对应官方文档确认可用能力和限制。

> **说明：** 不同产品型号与 NPU 架构版本的对应关系，以及具体硬件实现规格，应以对应版本的官方文档为准。本教程只介绍后续编程章节需要用到的抽象概念。


## 4. 本节小结

- 学习 AI Core 抽象硬件架构的目的，是为理解 SIMD 编程模型、分析数据流和后续编写算子建立统一硬件视角，而不是一开始记住所有产品规格。
- 抽象架构图可以先拆成计算单元、存储单元和搬运单元三类；Scalar 组织控制逻辑和任务发射，Vector/Cube 执行矢量或矩阵计算。
- Global Memory（GM）保存 Device 侧完整输入输出，Local Memory（LM）保存当前 AI Core、当前分片的数据和中间结果。
- DMA 负责在 GM 与片上存储、不同片上存储层级之间搬运数据；搬运和计算存在依赖时，需要同步信号保证先后关系正确。
- 从计算数据流阅读架构图，可以抓住“GM -> LM -> Vector/Cube -> GM”这条主线，先判断数据在哪里、由谁计算、结果写回哪里。
- 不同 NPU 架构版本在计算能力、存储层级、搬运路径和编程模型上会有差异；入门阶段应先掌握统一抽象，再结合目标架构查具体能力和限制。


## 5. 课后练习

请根据本节的关键内容完成以下单选题，建议先独立作答，再查看答案解析。

1. （单选题）学习 AI Core 抽象硬件架构的主要目的是什么？  
    A. 直接记住所有产品型号的硬件规格  
    B. 为理解 SIMD 编程模型和后续编写算子建立统一硬件视角  
    C. 只学习 Host 侧调用流程  
    D. 用抽象图替代目标架构的官方文档  

2. （单选题）在抽象架构中，哪个组件主要负责地址计算、流程控制和任务发射？  
    A. Scalar  
    B. Vector  
    C. Global Memory  
    D. DMA  

3. （单选题）在 SIMD 入门场景中，Vector 更适合负责哪类工作？  
    A. 保存完整输入输出数据  
    B. 在 GM 与 LM 之间搬运数据  
    C. 对一批元素执行相同或相近的逐元素计算  
    D. 记录产品型号和架构版本  

4. （单选题）关于 GM 和 LM 的关系，以下哪项理解最准确？  
    A. GM 通常保存 Device 侧全局可见的数据，LM 通常保存当前 AI Core 使用的片上分片数据和中间结果  
    B. LM 通常保存完整输入输出，GM 只保存当前分片  
    C. GM 和 LM 都是 Host 侧内存，AI Core 不能直接访问  
    D. GM 和 LM 的位置、容量和职责完全相同  

5. （单选题）从计算数据流角度看，SIMD 计算通常围绕哪条主线组织？  
    A. Host -> Kernel -> Host  
    B. GM -> LM -> Vector/Cube -> GM  
    C. Scalar -> Host -> GM -> Scalar  
    D. 架构版本 -> 产品型号 -> 编译选项  

**运行下方单元查看答案。**


In [ ]:
!cat ./answer/03.03.01_answer.txt
